In [31]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, KNNImputer, SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.neighbors import BallTree
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [32]:
training_df = pd.read_csv(
    filepath_or_buffer='training_faults_diagnostics.csv',
    low_memory=False
)
training_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1058069 entries, 0 to 1058068
Data columns (total 48 columns):
 #   Column                     Non-Null Count    Dtype  
---  ------                     --------------    -----  
 0   RecordID                   1058069 non-null  int64  
 1   EventTimeStamp             1058069 non-null  object 
 2   eventDescription           1002335 non-null  object 
 3   ecuSoftwareVersion         831493 non-null   object 
 4   ecuModel                   1002466 non-null  object 
 5   ecuMake                    1002466 non-null  object 
 6   ecuSource                  1058069 non-null  int64  
 7   spn                        1058069 non-null  int64  
 8   fmi                        1058069 non-null  int64  
 9   active                     1058069 non-null  bool   
 10  activeTransitionCount      1058069 non-null  int64  
 11  EquipmentID                1058069 non-null  object 
 12  MCTNumber                  1058069 non-null  int64  
 13  Latitude    

In [33]:
unnecessary_columns = [
    'RecordID',
    'EventTimeStamp',
    'eventDescription',
    'ecuSoftwareVersion',
    'ecuMake',
    'ecuModel',
    'ecuSource',
    'activeTransitionCount',
    'EquipmentID',
    'MCTNumber',
    'Latitude',
    'Longitude',
    'LocationTimeStamp',
    'NearServiceStation',
    'IsFullDerate',
    'Severity_Level_Numeric',
    'Derate_Target_2.0-0.001',
    'Derate_Target_8.0-0.001',
    'Derate_Target_12.0-0.001',
    'AcceleratorPedal',
    'CruiseControlSetSpeed',
    'CruiseControlActive',
    'DistanceLtd',
    'EngineTimeLtd',
    'FuelLevel',
    'FuelLtd',
    'IgnStatus',
    'LampStatus',
    'ParkingBrake'
]
len(unnecessary_columns)

29

In [34]:
features = [
    'spn',
    'fmi',
    'active',
    'Severity_Level',
    'Derate_Target_4.0-0.001',
    'BarometricPressure',
    'EngineCoolantTemperature',
    'EngineLoad',
    'EngineOilPressure',
    'EngineOilTemperature',
    'EngineRpm',
    'FuelRate',
    'FuelTemperature',
    'IntakeManifoldTemperature',
    'ServiceDistance',
    'Speed',
    'SwitchedBatteryVoltage',
    'Throttle',
    'TurboBoostPressure'
]
len(features)

19

In [35]:
# Conver SPN and FMI values to strings
training_df['spn'] = training_df['spn'].astype(str)
training_df['fmi'] = training_df['fmi'].astype(str)

In [36]:
# Create dataset with desired features
training_df = training_df[features]
training_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1058069 entries, 0 to 1058068
Data columns (total 19 columns):
 #   Column                     Non-Null Count    Dtype  
---  ------                     --------------    -----  
 0   spn                        1058069 non-null  object 
 1   fmi                        1058069 non-null  object 
 2   active                     1058069 non-null  bool   
 3   Severity_Level             401510 non-null   object 
 4   Derate_Target_4.0-0.001    1058069 non-null  int64  
 5   BarometricPressure         521532 non-null   float64
 6   EngineCoolantTemperature   521591 non-null   float64
 7   EngineLoad                 521085 non-null   float64
 8   EngineOilPressure          521733 non-null   float64
 9   EngineOilTemperature       519690 non-null   float64
 10  EngineRpm                  522191 non-null   float64
 11  FuelRate                   520788 non-null   float64
 12  FuelTemperature            272924 non-null   float64
 13  IntakeManifo

## Identify features for imputing missing values

In [37]:
# Drop columns that have too many NaN values
nan_drop_threshold = 0.8

drop_nan_columns = training_df.columns[training_df.isna().mean() > nan_drop_threshold]
print(drop_nan_columns)

training_df = training_df.drop(columns=drop_nan_columns)

Index(['ServiceDistance', 'SwitchedBatteryVoltage'], dtype='object')


In [38]:
# Group categorical columns
categorical_columns = training_df.select_dtypes(include=["object", "bool"]).columns
print(categorical_columns)
print(len(categorical_columns))

# Group numeric columns
numeric_columns = training_df.select_dtypes(include=["int64", "float64"]).columns
print(numeric_columns)
print(len(numeric_columns))

Index(['spn', 'fmi', 'active', 'Severity_Level'], dtype='object')
4
Index(['Derate_Target_4.0-0.001', 'BarometricPressure',
       'EngineCoolantTemperature', 'EngineLoad', 'EngineOilPressure',
       'EngineOilTemperature', 'EngineRpm', 'FuelRate', 'FuelTemperature',
       'IntakeManifoldTemperature', 'Speed', 'Throttle', 'TurboBoostPressure'],
      dtype='object')
13


In [39]:
# Group numeric columns by threshold
nan_low_threshold = 0.4

low_nan_numeric_columns = training_df[numeric_columns].columns[
    training_df[numeric_columns].isna().mean() <= nan_low_threshold
]
print(low_nan_numeric_columns)
print(len(low_nan_numeric_columns))

medium_nan_numeric_columns = training_df[numeric_columns].columns[
    training_df[numeric_columns].isna().mean() > nan_low_threshold
]
print(medium_nan_numeric_columns)
print(len(medium_nan_numeric_columns))

Index(['Derate_Target_4.0-0.001'], dtype='object')
1
Index(['BarometricPressure', 'EngineCoolantTemperature', 'EngineLoad',
       'EngineOilPressure', 'EngineOilTemperature', 'EngineRpm', 'FuelRate',
       'FuelTemperature', 'IntakeManifoldTemperature', 'Speed', 'Throttle',
       'TurboBoostPressure'],
      dtype='object')
12


## Split training dataset

In [40]:
target = 'Derate_Target_4.0-0.001'

In [41]:
X = training_df.drop(columns=[target])
y = training_df[target]

In [42]:
y.isna().sum()

np.int64(0)

In [43]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

## Create pipeline and fit model

In [44]:
categorical_pipe = Pipeline(
    steps=[
        ('categorical_imputer', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(handle_unknown='ignore'))
    ]
)

low_nan_numeric_pipe = Pipeline(
    steps=[
        ('scaler', StandardScaler()),
        ('low_nan_numeric_imputer', SimpleImputer(strategy='median'))
    ]
)

medium_nan_numeric_pipe = Pipeline(
    steps=[
        ('scaler', StandardScaler()),
        ('medium_nan_numeric_imputer', IterativeImputer(max_iter=20, random_state=30))
    ]
)

In [45]:
ct = ColumnTransformer(
    transformers=[
        ('categorical_pipe', categorical_pipe, categorical_columns),
        ('low_nan_numeric_pipe', low_nan_numeric_pipe, low_nan_numeric_columns.drop(target)),
        ('medium_nan_numeric_pipe', medium_nan_numeric_pipe, medium_nan_numeric_columns)
    ]
)

In [46]:
pipe = Pipeline(
    steps=[
        ('transformer', ct),
        ('model', MLPClassifier(
            activation='relu',
            hidden_layer_sizes=(64,64,64)
        ))
    ]
)

In [47]:
pipe.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/impute/_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


Pipeline(steps=[('transformer',
                 ColumnTransformer(transformers=[('categorical_pipe',
                                                  Pipeline(steps=[('categorical_imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('ohe',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  Index(['spn', 'fmi', 'active', 'Severity_Level'], dtype='object')),
                                                 ('low_nan_numeric_pipe',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler()),
                                                                  ('low_n...
                                                                  ('medium_nan_numeric_imputer',
                                                                   IterativeImputer(max_iter=20,
                                                                                    random_state=30))]),
                                                  Index(['BarometricPressure', 'EngineCoolantTemperature', 'EngineLoad',
       'EngineOilPressure', 'EngineOilTemperature', 'EngineRpm', 'FuelRate',
       'FuelTemperature', 'IntakeManifoldTemperature', 'Speed', 'Throttle',
       'TurboBoostPressure'],
      dtype='object'))])),
                ('model', MLPClassifier(hidden_layer_sizes=(64, 64, 64)))])

In [48]:
y_pred_train = pipe.predict(X_train)
y_pred_test = pipe.predict(X_test)

In [49]:
print(classification_report(
    y_true=y_train,
    y_pred=y_pred_train
))

print(classification_report(
    y_true=y_test,
    y_pred=y_pred_test
))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00    739951
           1       0.61      0.12      0.21       697

    accuracy                           1.00    740648
   macro avg       0.81      0.56      0.60    740648
weighted avg       1.00      1.00      1.00    740648

              precision    recall  f1-score   support

           0       1.00      1.00      1.00    317122
           1       0.56      0.11      0.18       299

    accuracy                           1.00    317421
   macro avg       0.78      0.56      0.59    317421
weighted avg       1.00      1.00      1.00    317421



In [50]:
training_cm = confusion_matrix(
    y_true=y_train,
    y_pred=y_pred_train
)
print(training_cm)

training_test_cm = confusion_matrix(
    y_true=y_test,
    y_pred=y_pred_test
)
print(training_test_cm)

[[739896     55]
 [   610     87]]
[[317096     26]
 [   266     33]]


In [51]:
print(f'Training Savings: {(training_cm[1][1]*4000) - (training_cm[1][0]*500)}')
print(f'Training Test Savings: {(training_test_cm[1][1]*4000) - (training_test_cm[1][0]*500)}')

Training Savings: 43000
Training Test Savings: -1000


In [52]:
testing_df = pd.read_csv(
    filepath_or_buffer='testing_faults_diagnostics.csv',
    low_memory=False
)

In [53]:
testing_target = testing_df[target]

In [54]:
testing_df = testing_df.drop(columns=unnecessary_columns)
testing_df = testing_df.drop(columns=drop_nan_columns)
testing_df = testing_df.drop(columns=target)
testing_df.columns

Index(['spn', 'fmi', 'active', 'Severity_Level', 'BarometricPressure',
       'EngineCoolantTemperature', 'EngineLoad', 'EngineOilPressure',
       'EngineOilTemperature', 'EngineRpm', 'FuelRate', 'FuelTemperature',
       'IntakeManifoldTemperature', 'Speed', 'Throttle', 'TurboBoostPressure'],
      dtype='object')

In [55]:
testing_df['prediction'] = pipe.predict(testing_df)

In [56]:
print(classification_report(
    y_true=testing_target,
    y_pred=testing_df['prediction']
))

              precision    recall  f1-score   support

           0       1.00      0.90      0.95    129144
           1       0.00      0.12      0.00       122

    accuracy                           0.90    129266
   macro avg       0.50      0.51      0.48    129266
weighted avg       1.00      0.90      0.95    129266



In [57]:
testing_cm = confusion_matrix(
    y_true=testing_target,
    y_pred=testing_df['prediction']
)
testing_cm

array([[116728,  12416],
       [   107,     15]])

In [58]:
print(f'Savings: {(testing_cm[1][1]*4000) - (testing_cm[1][0]*500)}')

Savings: 6500
